# 04 - Train ETA Model (Memory Optimized)

Train with 200K samples (float32) - fits in 12-13GB RAM

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/smart-ambulance-ml'
!pip install -q xgboost lightgbm joblib matplotlib

import numpy as np
import pandas as pd
import gc
import matplotlib.pyplot as plt
import joblib
import json
import time
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Load data
print("Loading ETA data...")
data = np.load(f'{PROJECT_DIR}/datasets/processed/eta_train.npz', allow_pickle=True)
X_train = data['X_train'].astype(np.float32)
X_val = data['X_val'].astype(np.float32)
X_test = data['X_test'].astype(np.float32)
y_train = data['y_train'].astype(np.float32)
y_val = data['y_val'].astype(np.float32)
y_test = data['y_test'].astype(np.float32)
feature_names = data['feature_names'].tolist()

print(f"✓ Loaded: Train {X_train.shape}, Val {X_val.shape}, Test {X_test.shape}")
print(f"RAM usage: ~{X_train.nbytes / 1e9:.2f}GB for training data")

In [ ]:
print("\n="*60)
print("TRAINING ETA MODELS")
print("="*60)

models = {
    'Random Forest': RandomForestRegressor(
        n_estimators=150, max_depth=16, min_samples_split=4,
        random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBRegressor(
        n_estimators=150, max_depth=6, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.85,
        random_state=42, n_jobs=-1,
        tree_method='hist'  # Memory efficient
    ),
    'LightGBM': LGBMRegressor(
        n_estimators=150, learning_rate=0.05, num_leaves=31,
        subsample=0.85, colsample_bytree=0.85,
        random_state=42, n_jobs=-1, verbose=-1
    )
}

results = {}
trained_models = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    start = time.time()
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    y_pred = model.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2 = r2_score(y_val, y_pred)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'time': time.time() - start}
    print(f"  ✓ MAE: {mae:.3f} min, RMSE: {rmse:.3f}, R2: {r2:.4f}")
    gc.collect()

print("\n" + pd.DataFrame(results).T.round(4).to_string())

In [ ]:
print("\n="*60)
print("TRAINING STACKING REGRESSOR")
print("="*60)

base_estimators = [(name, model) for name, model in trained_models.items()]
meta_learner = ElasticNet(alpha=0.01, l1_ratio=0.3, random_state=42, max_iter=1000)

stacking_model = StackingRegressor(
    estimators=base_estimators,
    final_estimator=meta_learner,
    cv=2,  # Reduced from 3
    n_jobs=-1
)

print("Training (2-3 minutes)...")
start = time.time()
stacking_model.fit(X_train, y_train)

y_pred = stacking_model.predict(X_val)
mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
r2 = r2_score(y_val, y_pred)

results['Stacking'] = {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'time': time.time() - start}
trained_models['Stacking'] = stacking_model

print(f"✓ Stacking: MAE={mae:.3f} min, RMSE={rmse:.3f}, R2={r2:.4f}")
gc.collect()

In [ ]:
print("\n="*60)
print("TEST EVALUATION")
print("="*60)

test_results = {}
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    test_results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}

test_df = pd.DataFrame(test_results).T.sort_values('MAE')
print(test_df.round(4))

print(f"\n✓ Best: {test_df.index[0]} (MAE={test_df.iloc[0]['MAE']:.3f} min)")

In [ ]:
# Quick visualization
y_pred_best = trained_models[test_df.index[0]].predict(X_test[:2000])
y_sample = y_test[:2000]

plt.figure(figsize=(8, 6))
plt.scatter(y_sample, y_pred_best, alpha=0.4, s=10)
plt.plot([y_sample.min(), y_sample.max()], [y_sample.min(), y_sample.max()], 'r--', lw=2)
plt.xlabel('Actual (min)')
plt.ylabel('Predicted (min)')
plt.title(f'ETA Predictions - {test_df.index[0]}\nMAE: {test_df.iloc[0]["MAE"]:.2f} min')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/eta_predictions.png', dpi=150)
plt.show()

In [ ]:
print("\n="*60)
print("SAVING MODELS")
print("="*60)

models_dir = f'{PROJECT_DIR}/models/eta'
os.makedirs(models_dir, exist_ok=True)

for name, model in trained_models.items():
    filename = name.lower().replace(' ', '_') + '.joblib'
    joblib.dump(model, f'{models_dir}/{filename}')
    print(f"✓ Saved {name}")

# Update registry
try:
    with open(f'{PROJECT_DIR}/models/model_registry.json', 'r') as f:
        registry = json.load(f)
except:
    registry = {}

registry['eta'] = {
    'models': list(trained_models.keys()),
    'best_model': test_df.index[0],
    'test_mae': test_results[test_df.index[0]]['MAE'],
    'test_r2': test_results[test_df.index[0]]['R2']
}

with open(f'{PROJECT_DIR}/models/model_registry.json', 'w') as f:
    json.dump(registry, f, indent=2)

print(f"\n✓ ETA training complete!")
gc.collect()